# 随机梯度下降（SGD）手撕实现

> 本文件原为空，按"详细解析 + 骨架"补全。

## 1. 原理
- **批量梯度下降（BGD）**：用全部样本算梯度，更新稳但慢、显存大。
- **随机梯度下降（SGD）**：每次用 1 个（或一个 mini-batch 的）样本估计梯度，引入噪声但更新快、能跳出鞍点。
- 更新规则：$\theta_{t+1}=\theta_t-\eta\, g_t$，$g_t=\nabla \ell(\theta_t;x_{i_t})$。
- mini-batch 是工程默认：在噪声与效率间折中，且利于 GPU 并行。

## 2. 学习率
太大震荡发散，太小收敛慢。常用策略：常数、`1/t` 衰减、warmup+cosine。带动量：
$$v_t=\beta v_{t-1}+g_t,\quad \theta_{t+1}=\theta_t-\eta v_t$$
动量平滑噪声、加速在一致梯度方向上的收敛。

## 3. 考察点
- mini-batch 采样与 epoch 概念
- 与 `torch.optim.SGD` 对拍
- 动量 / weight decay / Nesterov 的实现差异
- 为什么 `zero_grad` 每步要清零（PyTorch 梯度默认累加）

In [ ]:
import torch

# 任务：拟合 y = 2x + 3 的线性回归 y = w*x + b
torch.manual_seed(0)
x = torch.linspace(-1, 1, 100)
y = 2 * x + 3 + 0.1 * torch.randn(100)

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# ---- 手撕 SGD 训练循环（骨架）----
lr = 0.1
batch_size = 16
epochs = 50
n = x.size(0)

for epoch in range(epochs):
    perm = torch.randperm(n)                       # 每个 epoch 打乱
    for i in range(0, n, batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = x[idx], y[idx]
        # TODO: 前向 pred = w*xb + b ; loss = ((pred-yb)**2).mean()
        # TODO: loss.backward()
        # TODO: with torch.no_grad(): w -= lr*w.grad ; b -= lr*b.grad
        # TODO: w.grad.zero_() ; b.grad.zero_()
        raise NotImplementedError

print('w=', w.item(), 'b=', b.item())  # 期望 w≈2, b≈3

In [ ]:
# 对照：用 torch.optim.SGD 拟合，验证结果一致
w2 = torch.zeros(1, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([w2, b2], lr=lr)
torch.manual_seed(0)
for epoch in range(epochs):
    perm = torch.randperm(n)
    for i in range(0, n, batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = x[idx], y[idx]
        opt.zero_grad()
        loss = ((w2 * xb + b2 - yb) ** 2).mean()
        loss.backward()
        opt.step()
print('torch: w=', w2.item(), 'b=', b2.item())

## 小结
- SGD = 用 mini-batch 随机梯度做下降更新；`zero_grad` 必须每步清零，否则梯度跨 batch 累加（这恰是 gradient accumulation 利用的特性）。
- 动量/Adam 在 SGD 基础上改"梯度聚合方式"，更新公式仍为 $\theta\leftarrow\theta-\eta\,\text{update}$。
- 面试常问：为什么 SGD 能收敛？（凸下期望梯度=真梯度；非凸下收敛到驻点附近，噪声帮助逃鞍点）。